In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "enviroment_bj").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("project_root:", ROOT)

## Data for example 

In [ ]:
from data.syntethic_long_context_retrieval import *

CFG = SyntheticRetrievalConfig(
    # Dataset size
    num_train_examples=50_000,
    num_val_examples=5_000,

    # Sequence length
    block_size=64,              # debe ser <= model.config.max_seq_len
    min_filler_tokens=8,
    max_filler_tokens=32,

    # Task structure
    num_keys_per_example=4,
    vocab_filler_size=68,      
    num_key_types=64,
    num_value_types=64,

    # DataLoader
    batch_size=32,
    num_workers=0,

    # Reproducibility
    seed=42)

train_loader, val_loader, tokenizer = create_synthetic_retrieval_dataloaders(
        cfg=CFG,
        use_mtp=False,)

x, y = next(iter(train_loader))

## Instance Model 

In [ ]:
from src.mini_deepseek_class import * 

cfg = DeepSeekV4LMConfig(
    vocab_size=216,
    d_model=64,
    n_layers=4,
    max_seq_len=64,
    pad_token_id=0,
    ignore_index=-100,

    # ========================================================
    # Attention: hybrid CSA/HCA baseline
    # Pattern:
    #   layer 0 -> CSA
    #   layer 1 -> HCA
    #   layer 2 -> CSA
    #   layer 3 -> HCA
    # ========================================================
    attention_type="hybrid",
    attention_pattern=("csa", "hca"),

    n_heads=4,
    head_dim=16,
    rotary_dim=16,

    attention_dropout=0.0,
    residual_dropout=0.0,
    use_attention_bias=False,
    use_rope=True,
    rope_theta=10000.0,

    # ========================================================
    # CSA config
    # ========================================================
    compression_factor=4,
    top_k_blocks=2,
    window_size=8,
    indexer_dim=8,
    n_indexer_heads=2,
    query_compression_dim=16,

    use_attention_sink=True,
    use_grouped_output_projection=True,
    output_projection_groups=2,
    use_indexer_score_bias=False,
    use_separate_local_kv=True,

    # ========================================================
    # HCA config
    # HCA should compress more aggressively than CSA.
    # With max_seq_len=64, hca_compression_factor=8 is a sane
    # mini baseline. 16 also works but gives very few global blocks.
    # ========================================================
    hca_compression_factor=8,

    # ========================================================
    # FFN: DeepSeekMoE
    # ========================================================
    ffn_type="moe",
    num_experts=4,
    top_k_experts=2,

    expert_hidden_dim=128,
    expert_expansion_factor=4.0,
    expert_multiple_of=1,

    shared_experts=1,
    shared_hidden_dim=128,
    shared_expansion_factor=4.0,

    router_type="learned",
    router_score_fn="sqrt_softplus",
    normalize_topk_weights=True,
    topk_weight_scale=1.0,
    router_jitter_noise=0.0,
    hash_routing_stride=1,

    routed_scale=1.0,
    shared_scale=1.0,

    balance_loss_weight=0.01,
    sequence_balance_loss_weight=0.01,

    # ========================================================
    # mHC
    # ========================================================
    use_mhc=True,
    n_hc=4,
    mhc_sinkhorn_iters=20,
    mhc_eps=1e-6,
    mhc_dynamic=True,
    mhc_expand_mode="first",
    mhc_collapse_mode="readout",

    mhc_use_log_sinkhorn=False,
    mhc_sinkhorn_fp32=True,
    mhc_init_alpha=1e-3,
    mhc_alpha_max=1.0,
    mhc_bounded_alpha=True,

    # ========================================================
    # MTP
    # ========================================================
    use_mtp=True,
    mtp_depth=2,
    mtp_hidden_dim=64,
    use_mtp_transform=True,
    mtp_activation="silu",
    mtp_dropout=0.0,
    mtp_loss_weight=0.3,
    mtp_tie_with_lm_head=False,
    mtp_depth_loss_weights=None,
    mtp_validate_label_range=True,

    # ========================================================
    # Embedding / norm / init
    # ========================================================
    embedding_dropout=0.0,
    scale_embeddings=False,
    tie_word_embeddings=True,

    rms_norm_eps=1e-6,
    init_std=0.02,

    # ========================================================
    # Loss semantics
    #   input_ids = ids[:-1]
    #   labels    = ids[1:]
    # ========================================================
    labels_are_shifted=True,
    ignore_pad_token_in_loss=True,)

model = DeepSeekV4LM(cfg)
model.to('cuda')

---

# Autocast and Optimizer

In [ ]:
import numpy as np

from training.train_deepseek import *

precision = setup_device_and_precision(
    device="auto",
    amp_enabled=True,
    amp_dtype="bf16",
)

device = precision["device"]
model = model.to(device)

optimizer, opt_info = build_adamw_optimizer(
    model=model,
    learning_rate=3e-4,
    weight_decay=0.1,
    betas=(0.9, 0.95),
    eps=1e-8,
    verbose=True,
)


batch = next(iter(train_loader))
batch = normalize_lm_batch(batch)
batch = move_batch_to_device(batch, device)


with autocast_ctx(
    device=device,
    enabled=precision["amp_enabled"],
    amp_dtype=precision["amp_dtype_requested"],
    cache_enabled=precision["cache_enabled"],
    fallback_bf16_to_fp16=precision["fallback_bf16_to_fp16"],
):
    outputs = model(**batch)
    loss = outputs["loss"]

print(loss)

scaler = precision["scaler"]

if scaler is not None:
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
else:
    loss.backward()
    optimizer.step()




Synthetic tokenizer vocab size: 216
Tokenizer vocab: 216
Model vocab: 216
AdamW parameter groups
Decay tensors     : 170
No-decay tensors  : 127
Decay params      : 28,851,712
No-decay params   : 452,508
Total params      : 29,304,220
tensor(7.0348, device='cuda:0', grad_fn=<AddBackward0>)


---

# Chekpoint

In [96]:
ckpt_path = save_checkpoint(
    checkpoint_dir="checkpoints/deepseekv4_mini",
    model=model,
    optimizer=optimizer,
    scheduler=None,
    scaler=precision["scaler"],
    epoch=0,
    step=1,
    best_metric=None,
    config=getattr(model, "config", None),
    extra_state={
        "loss": float(loss.detach().cpu().item()),
        "tokenizer_vocab_size": tokenizer.vocab_size,
    },
    keep_last_n=3,
)

print("Saved:", ckpt_path)

Saved: checkpoints/deepseekv4_mini/checkpoint_step_00000001.pt


In [97]:
state = load_checkpoint(
    checkpoint_path=ckpt_path,
    model=model,
    optimizer=optimizer,
    scheduler=None,
    scaler=precision["scaler"],
    map_location=device,
    strict=True,
)

print(state)

{'epoch': 0, 'step': 1, 'best_metric': None, 'config': {'vocab_size': 216, 'd_model': 64, 'n_layers': 2, 'max_seq_len': 64, 'pad_token_id': 0, 'ignore_index': -100, 'labels_are_shifted': True, 'ignore_pad_token_in_loss': True, 'embedding_dropout': 0.0, 'scale_embeddings': False, 'tie_word_embeddings': True, 'rms_norm_eps': 1e-06, 'attention_type': 'csa', 'n_heads': 4, 'head_dim': 16, 'attention_dropout': 0.0, 'residual_dropout': 0.0, 'use_attention_bias': False, 'use_rope': True, 'rope_theta': 10000.0, 'rotary_dim': 16, 'compression_factor': 4, 'hca_compression_factor': 16, 'window_size': 8, 'top_k_blocks': 2, 'indexer_dim': 8, 'n_indexer_heads': 2, 'query_compression_dim': 16, 'use_attention_sink': True, 'use_grouped_output_projection': True, 'output_projection_groups': 2, 'use_indexer_score_bias': False, 'use_separate_local_kv': True, 'ffn_type': 'moe', 'mlp_hidden_dim': None, 'mlp_expansion_factor': 4.0, 'mlp_multiple_of': 1, 'mlp_dropout': 0.0, 'use_mlp_bias': False, 'num_experts':

---

# EMA

In [99]:
ema = EMA(
    model=model,
    decay=0.999,
    device="cpu",
    use_num_updates=True,
    update_after_step=1,
    update_every=1,
)

optimizer.zero_grad(set_to_none=True)

with autocast_ctx(
    device=device,
    enabled=precision["amp_enabled"],
    amp_dtype=precision["amp_dtype_requested"],
    cache_enabled=precision["cache_enabled"],
    fallback_bf16_to_fp16=precision["fallback_bf16_to_fp16"],
):
    outputs = model(**batch)
    loss = outputs["loss"]

scaler = precision["scaler"]

if scaler is not None:
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
else:
    loss.backward()
    optimizer.step()

ema.update(model, step=1)

ok, status, rel = ema_health(ema, model)
print("EMA:", ok, status, rel)

EMA: True ok 0.0009718340313424267


---
# Metrics

In [101]:
val_metrics = evaluate_lm(
    model=model,
    dataloader=val_loader,
    device=device,
    precision=precision,
    max_batches=50,
    topk=(1, 5),
    ema=ema,
    use_ema=False,
)

print(format_metrics(val_metrics, prefix="val"))

val/loss=5.3535 | val/perplexity=211.3408 | val/token_accuracy=0.0074 | val/top_5_accuracy=0.0665 | val/sequence_accuracy=0.0000 | val/mean_confidence=0.0145 | val/mean_true_token_prob=0.0048 | val/mean_entropy=5.3563 | val/valid_tokens=86117


---

# Special Metrics for DeepSeekv4 modules

In [162]:
outputs = model(
    input_ids=batch["input_ids"],
    labels=batch.get("labels", None),
    mtp_labels=batch.get("mtp_labels", None),
    attention_mask=batch.get("attention_mask", None),
    position_ids=batch.get("position_ids", None),
    return_aux=True,
    need_weights=False,
)

loss = outputs["loss"]

deepseek_metrics = compute_deepseek_module_metrics(
    outputs=outputs,
    model=model,
    prefix="train",
)

print(deepseek_metrics)

{'train/loss': 6.985154151916504, 'train/lm_loss': 5.372533321380615, 'train/mtp_loss': 1.612534999847412, 'train/moe_aux_loss': 8.572459046263248e-05, 'train/raw_mtp_loss': 5.375116348266602, 'train/weighted_mtp_loss': 1.612534999847412, 'train/balance_loss': 8.863806669978658e-06, 'train/sequence_balance_loss': 3.399848901608493e-05, 'train/perplexity_from_lm_loss': 215.40787444813452, 'train/loss_minus_components': 1.0609801393002272e-07, 'train/moe/expert_fraction_mean': 0.25, 'train/moe/expert_fraction_std': 0.029772145673632622, 'train/moe/expert_fraction_min': 0.195068359375, 'train/moe/expert_fraction_max': 0.294189453125, 'train/moe/dead_experts_across_layers': 0.0, 'train/moe/active_experts_across_layers': 8.0, 'train/moe/sequence_expert_fraction_mean': 0.25, 'train/moe/sequence_expert_fraction_std': 0.05830822512507439, 'train/moe/sequence_expert_fraction_min': 0.0859375, 'train/moe/sequence_expert_fraction_max': 0.3828125, 'train/moe/n_experts_used_per_batch_mean': 4.0, 'tr

In [163]:
print_deepseek_module_metrics(
    deepseek_metrics,
    prefix="train",
    title="DeepSeek-V4 module diagnostics",
)


DeepSeek-V4 module diagnostics

1. Loss / convergence metrics
-----------------------------
  loss                    : 6.9852       # lower is better; total optimized objective
  lm_loss                 : 5.3725       # lower is better; main next-token CE
  perplexity_from_lm_loss : 215.4079     # lower is better; exp(lm_loss)
  mtp_loss                : 1.6125       # auxiliary; should not dominate total loss
  moe_aux_loss            : 8.5725e-05   # should be small; high values imply routing imbalance penalty
  loss_minus_components   : 1.0610e-07   # should be ≈ 0; nonzero means loss accounting bug
  raw_mtp_loss            : 5.3751       # unweighted MTP CE; compare with lm_loss
  weighted_mtp_loss       : 1.6125       # raw_mtp_loss * mtp_loss_weight

2. MoE routing health — critical
--------------------------------
  router_entropy_mean           : 1.3850   # higher = more uniform routing
  expert_fraction_min           : 0.1951   # too close to 0 => dead/underused expert
  ex

---

# MUON + HYBRID MUON-ADAMW OPTIMIZER

In [103]:
optimizer1, opt_info1 = build_muon_adamw_optimizer(
    model=model,
    learning_rate=3e-4,
    weight_decay=0.1,
    betas=(0.9, 0.95),
    eps=1e-8,
    muon_lr=None,              # None => usa learning_rate
    muon_momentum=0.95,
    muon_nesterov=True,
    muon_ns_steps=5,
    muon_eps=1e-7,
    muon_weight_decay=0.0,     # recomendado inicial
    verbose=True,
)

Hybrid Muon + AdamW parameter groups
Muon tensors          : 66
AdamW decay tensors   : 4
AdamW no-decay tensors: 18
--------------------------------------------------------------------------------
Muon params           : 285,952
AdamW params          : 42,368
Total trainable params: 328,320


In [104]:
def audit_muon_assignment(model, opt_info):
    muon = set(opt_info["muon_names"])
    adamw_decay = set(opt_info["adamw_decay_names"])
    adamw_no_decay = set(opt_info["adamw_no_decay_names"])
    adamw = adamw_decay | adamw_no_decay

    # Detect tied lm_head.
    lm_head_tied = False
    try:
        emb_weight = model.embedding.token_embedding.weight
        lm_weight = model.lm_head.weight
        lm_head_tied = emb_weight.data_ptr() == lm_weight.data_ptr()
    except Exception:
        pass

    embedding_in_adamw = any(
        "embedding" in n or "token_embedding" in n
        for n in adamw
    )

    lm_head_in_adamw_by_name = any("lm_head" in n for n in adamw)
    lm_head_in_muon_by_name = any("lm_head" in n for n in muon)

    # If tied, it is okay that lm_head does not appear by name,
    # as long as the shared embedding parameter is in AdamW.
    lm_head_effectively_in_adamw = (
        lm_head_in_adamw_by_name
        or (lm_head_tied and embedding_in_adamw)
    )

    checks = {
        "embedding_in_adamw": embedding_in_adamw,
        "lm_head_tied_to_embedding": lm_head_tied,
        "lm_head_in_adamw_by_name": lm_head_in_adamw_by_name,
        "lm_head_effectively_in_adamw": lm_head_effectively_in_adamw,
        "lm_head_not_in_muon": not lm_head_in_muon_by_name,
        "mtp_vocab_heads_in_adamw": any("mtp_head.heads" in n for n in adamw),
        "norms_in_adamw": any("norm" in n.lower() for n in adamw),
        "attention_linears_in_muon": any(".attention." in n and n.endswith(".weight") for n in muon),
        "moe_experts_in_muon": any(".ffn.experts." in n and n.endswith(".weight") for n in muon),
        "shared_experts_in_muon": any(".ffn.shared_experts." in n and n.endswith(".weight") for n in muon),
        "router_in_muon": any(".ffn.router.weight" in n for n in muon),
        "mtp_transforms_in_muon": any("mtp_head.transforms" in n and n.endswith(".weight") for n in muon),
    }

    for k, v in checks.items():
        print(f"{k}: {v}")

    return checks

audit_muon_assignment(model , opt_info1)

embedding_in_adamw: True
lm_head_tied_to_embedding: True
lm_head_in_adamw_by_name: False
lm_head_effectively_in_adamw: True
lm_head_not_in_muon: True
mtp_vocab_heads_in_adamw: True
norms_in_adamw: True
attention_linears_in_muon: True
moe_experts_in_muon: True
shared_experts_in_muon: True
router_in_muon: True
mtp_transforms_in_muon: True


{'embedding_in_adamw': True,
 'lm_head_tied_to_embedding': True,
 'lm_head_in_adamw_by_name': False,
 'lm_head_effectively_in_adamw': True,
 'lm_head_not_in_muon': True,
 'mtp_vocab_heads_in_adamw': True,
 'norms_in_adamw': True,
 'attention_linears_in_muon': True,
 'moe_experts_in_muon': True,
 'shared_experts_in_muon': True,
 'router_in_muon': True,
 'mtp_transforms_in_muon': True}

In [145]:
batch = next(iter(train_loader))

batch = normalize_lm_batch(batch)
batch = move_batch_to_device(batch, device)

optimizer1.zero_grad(set_to_none=True)

with autocast_ctx(
    device=device,
    enabled=precision["amp_enabled"],
    amp_dtype=precision["amp_dtype_requested"],
    cache_enabled=precision["cache_enabled"],
    fallback_bf16_to_fp16=precision["fallback_bf16_to_fp16"],
):
    outputs = model(**batch)
    loss = outputs["loss"]

print(loss)

scaler = precision["scaler"]

if scaler is not None:
    # Para fp16 + HybridMuonAdamW habría que hacer step separado.
    # Para nuestro caso recomendado: bf16 => scaler=None.
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer1.muon)
    scaler.unscale_(optimizer1.adamw)
    scaler.step(optimizer1.muon)
    scaler.step(optimizer1.adamw)
    scaler.update()
else:
    loss.backward()
    optimizer1.step()

print("muon-adamw step ok")

tensor(9.1323, device='cuda:0', grad_fn=<NllLossBackward0>)
muon-adamw step ok


---

# Scheduler

In [166]:
optimizer, opt_info = build_adamw_optimizer(
    model=model,
    learning_rate=3e-4,
    weight_decay=0.1,
    betas=(0.9, 0.95),
    eps=1e-8,
    verbose=True,
)

scheduler = build_warmup_cosine_scheduler(
    optimizer=optimizer,
    total_steps=10_000,
    warmup_steps=500,
    min_lr=3e-5,
)

AdamW parameter groups
Decay tensors     : 66
No-decay tensors  : 75
Decay params      : 269,632
No-decay params   : 84,496
Total params      : 354,128


In [167]:
optimizer, opt_info = build_muon_adamw_optimizer(
    model=model,
    learning_rate=3e-4,
    weight_decay=0.1,
    betas=(0.9, 0.95),
    eps=1e-8,
    muon_lr=None,
    muon_momentum=0.95,
    muon_nesterov=True,
    muon_ns_steps=5,
    muon_eps=1e-7,
    muon_weight_decay=0.0,
    verbose=True,
)

scheduler = build_warmup_cosine_scheduler(
    optimizer=optimizer,
    total_steps=10_000,
    warmup_steps=500,
    min_lr=3e-5,
    min_muon_lr=None,  # None => usa min_lr también para Muon
)

Hybrid Muon + AdamW parameter groups
Muon tensors          : 66
AdamW decay tensors   : 4
AdamW no-decay tensors: 71
--------------------------------------------------------------------------------
Muon params           : 285,952
AdamW params          : 68,176
Total trainable params: 354,128


In [168]:
print("Initial LR:", scheduler.get_lr_dict())

for _ in range(5):
    scheduler.step()
    print(scheduler.get_lr_dict())

Initial LR: {'step': 0, 'muon_lr': 0.0003, 'adamw_lr': 0.0003, 'muon_lrs': [0.0003], 'adamw_lrs': [0.0003, 0.0003]}
{'step': 1, 'muon_lr': 6e-07, 'adamw_lr': 6e-07, 'muon_lrs': [6e-07], 'adamw_lrs': [6e-07, 6e-07]}
{'step': 2, 'muon_lr': 1.2e-06, 'adamw_lr': 1.2e-06, 'muon_lrs': [1.2e-06], 'adamw_lrs': [1.2e-06, 1.2e-06]}
{'step': 3, 'muon_lr': 1.8e-06, 'adamw_lr': 1.8e-06, 'muon_lrs': [1.8e-06], 'adamw_lrs': [1.8e-06, 1.8e-06]}
{'step': 4, 'muon_lr': 2.4e-06, 'adamw_lr': 2.4e-06, 'muon_lrs': [2.4e-06], 'adamw_lrs': [2.4e-06, 2.4e-06]}
{'step': 5, 'muon_lr': 2.9999999999999997e-06, 'adamw_lr': 2.9999999999999997e-06, 'muon_lrs': [2.9999999999999997e-06], 'adamw_lrs': [2.9999999999999997e-06, 2.9999999999999997e-06]}
